# 中证800 V87：V46 feature_fraction 随机子空间鲁棒性实验

目标：在完全冻结 V46 基础参数、特征选择、训练轮数和 OOS 划分的前提下，只比较 `feature_fraction=1.0` 与 `0.8`。

- 5 个年度 walk-forward OOS 折：2022、2023、2024、2025、2026（按数据可用月份）。
- 4 个随机子空间强度：`1.0 / 0.9 / 0.8 / 0.7`；3 个随机种子控制 bagging、feature fraction 与 LightGBM 随机状态。
- 严格 `label_end_safe`：训练标签的 `next_date` 必须不晚于 cutoff。
- 每个 fold 的特征筛选、填充值和 LightGBM Dataset 在两个 feature fraction 间完全共享。
- 不做逐笔回测；使用 RankIC、Top8 board-cap edge、Precision/Recall/NDCG/AUC 和选股重合度快速判断。

预注册通过条件：`ff080` 平均 RankIC delta > 0、Top8 edge delta >= 0，至少 2/3 种子和 3/5 有效折的 RankIC delta > 0，且最差种子 RankIC delta >= -0.005。

## 0. 导入、输出目录与进度条

In [ ]:
import os
import gc
import json
import warnings
import builtins as _bi
import datetime as _dt
from pathlib import Path

try:
    from jqdata import *
except Exception:
    pass

import lightgbm as lgb
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


PROJECT_DIR = Path.cwd()
OUT_DIR = PROJECT_DIR / "csi800_ml_v87_feature_fraction_robustness_outputs"
FIGURE_DIR = OUT_DIR / "figures"
for _path in [OUT_DIR, FIGURE_DIR]:
    _path.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
print("OUT_DIR:", OUT_DIR)

## 1. 冻结实验配置

In [ ]:
DATA_PATH_OVERRIDE = None
DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement.csv"),
]

STOCK_COL = "stock"
DATE_COL = "rebalance_date"
NEXT_DATE_COL = "next_date"
TARGET_COL = "alpha_1m"
INDUSTRY_COL = "industry_bucket"

FEATURE_FRACTIONS = [1.0, 0.9, 0.8, 0.7]
SEEDS = [17, 42, 101]
FIXED_ITER = 120
CORR_THRESHOLD = 0.70
MIN_TRAIN_MONTHS = 30
TRUE_TOP_N = 20
PRED_TOP_K = 8
BOARD_CAPS = {"chinext": 3, "star": 2}
NUM_THREADS = 4

FOLD_SPECS = [
    {"fold_id": "oos_2022", "cutoff": "2021-12-31", "test_start": "2022-01-01", "test_end": "2022-12-31"},
    {"fold_id": "oos_2023", "cutoff": "2022-12-31", "test_start": "2023-01-01", "test_end": "2023-12-31"},
    {"fold_id": "oos_2024", "cutoff": "2023-12-31", "test_start": "2024-01-01", "test_end": "2024-12-31"},
    {"fold_id": "oos_2025", "cutoff": "2024-12-31", "test_start": "2025-01-01", "test_end": "2025-12-31"},
    {"fold_id": "oos_2026", "cutoff": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-12-31"},
]

# Keep both fractions in smoke mode so the paired comparison remains valid.
SMOKE_TEST = False
SMOKE_MAX_FOLDS = 1
SMOKE_SEEDS = [42]

MIN_SEED_WINS = 2
MIN_FOLD_WINS = 3
MAX_WORST_SEED_RANK_IC_DELTA = -0.005

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

if SMOKE_TEST:
    FOLD_SPECS = FOLD_SPECS[:SMOKE_MAX_FOLDS]
    SEEDS = list(SMOKE_SEEDS)

print("feature fractions:", FEATURE_FRACTIONS)
print("seeds:", SEEDS)
print("folds:", [x["fold_id"] for x in FOLD_SPECS])
print("models planned:", len(FEATURE_FRACTIONS) * len(SEEDS) * len(FOLD_SPECS))

## 2. V46 特征与兼容工具

In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]
HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]
FULL_V46_COLS = unique_keep_order(BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS)


def fraction_tag(value):
    return "ff%03d" % int(round(float(value) * 100.0))


FRACTION_TAGS = [fraction_tag(value) for value in FEATURE_FRACTIONS]


def safe_rank_ic(a, b):
    d = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 3 or d["a"].nunique() < 2 or d["b"].nunique() < 2:
        return np.nan
    return d["a"].rank(method="average").corr(d["b"].rank(method="average"))


def safe_mean(values):
    s = pd.Series(list(values), dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    return float(s.mean()) if len(s) else np.nan


def binary_rank_auc(y_true, scores):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "score": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    positive = d["y"] > 0
    n_pos = int(positive.sum())
    n_neg = int(len(d) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = d["score"].rank(method="average")
    rank_sum = float(ranks[positive].sum())
    return (rank_sum - n_pos * (n_pos + 1) / 2.0) / float(n_pos * n_neg)


def binary_ndcg_at_k(y_true, scores, k):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "score": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna().sort_values("score", ascending=False)
    k_eff = _bi.min(int(k), len(d))
    if k_eff <= 0:
        return np.nan
    gains = (d["y"].head(k_eff).values > 0).astype(float)
    dcg = float((gains / np.log2(np.arange(k_eff, dtype=float) + 2.0)).sum())
    ideal_hits = _bi.min(k_eff, int((d["y"] > 0).sum()))
    if ideal_hits <= 0:
        return np.nan
    idcg = float((np.ones(ideal_hits) / np.log2(np.arange(ideal_hits, dtype=float) + 2.0)).sum())
    return dcg / idcg if idcg > 0 else np.nan


def stock_board(stock):
    code_value = str(stock).split(".")[0]
    if code_value.startswith(("300", "301")):
        return "chinext"
    if code_value.startswith(("688", "689")):
        return "star"
    return "other"


def board_capped_indices(sorted_df, target_n=PRED_TOP_K):
    selected = []
    counts = {}
    for idx, row in sorted_df.iterrows():
        board = stock_board(row[STOCK_COL])
        cap = BOARD_CAPS.get(board)
        if cap is not None and counts.get(board, 0) >= int(cap):
            continue
        selected.append(idx)
        counts[board] = counts.get(board, 0) + 1
        if len(selected) >= int(target_n):
            return selected
    for idx in sorted_df.index:
        if idx not in selected:
            selected.append(idx)
            if len(selected) >= int(target_n):
                break
    return selected

## 3. 数据加载、折计划与训练矩阵

In [ ]:
def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        path = Path(DATA_PATH_OVERRIDE)
        if path.exists():
            return path
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % path)
    for item in DATA_CANDIDATES:
        path = Path(item)
        if path.exists():
            return path
    raise IOError("training CSV not found: %s" % [str(Path(x)) for x in DATA_CANDIDATES])


def load_dataset(path):
    header = pd.read_csv(path, nrows=0)
    required = [STOCK_COL, DATE_COL, NEXT_DATE_COL, TARGET_COL, INDUSTRY_COL] + FULL_V46_COLS
    missing = [c for c in required if c not in header.columns]
    if missing:
        raise ValueError("dataset missing columns: %s" % ",".join(missing))
    df = pd.read_csv(path, usecols=required)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce").dt.normalize()
    df[NEXT_DATE_COL] = pd.to_datetime(df[NEXT_DATE_COL], errors="coerce").dt.normalize()
    df[STOCK_COL] = df[STOCK_COL].astype(str)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    for col in progress_iter(FULL_V46_COLS, total=len(FULL_V46_COLS), desc="compact V46 features"):
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(np.float32)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=[STOCK_COL, DATE_COL, NEXT_DATE_COL, TARGET_COL])
    return df.sort_values([DATE_COL, STOCK_COL]).reset_index(drop=True)


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            value = corr.iloc[i, j]
            if not pd.isnull(value) and abs(value) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    visited = set()
    components = []
    for col in feature_cols:
        if col in visited:
            continue
        stack = [col]
        component = []
        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)
            component.append(current)
            stack.extend(graph[current])
        components.append(component)
    return components


def select_features_train_only(train_df):
    missing = train_df[FULL_V46_COLS].isnull().sum().to_dict()
    keep = []
    remove = []
    for component in build_corr_components(train_df, FULL_V46_COLS, CORR_THRESHOLD):
        ordered = _bi.sorted(component, key=lambda x: (missing[x], x))
        keep.append(ordered[0])
        remove.extend(ordered[1:])
    return keep, remove


def make_fold_frames(df, spec):
    cutoff = pd.Timestamp(spec["cutoff"])
    test_start = pd.Timestamp(spec["test_start"])
    test_end = pd.Timestamp(spec["test_end"])
    train = df[(df[DATE_COL] <= cutoff) & (df[NEXT_DATE_COL] <= cutoff)].copy()
    test = df[(df[DATE_COL] >= test_start) & (df[DATE_COL] <= test_end)].copy()
    return train, test


DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)
print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape, "months:", df_all[DATE_COL].nunique())
print("dates:", df_all[DATE_COL].min(), "->", df_all[DATE_COL].max())

fold_plan_rows = []
valid_fold_specs = []
for spec in progress_iter(FOLD_SPECS, total=len(FOLD_SPECS), desc="build fold plan"):
    train_df, test_df = make_fold_frames(df_all, spec)
    row = dict(spec)
    row["train_rows"] = int(len(train_df))
    row["train_months"] = int(train_df[DATE_COL].nunique())
    row["test_rows"] = int(len(test_df))
    row["test_months"] = int(test_df[DATE_COL].nunique())
    row["max_train_next_date"] = str(train_df[NEXT_DATE_COL].max().date()) if len(train_df) else ""
    row["status"] = "run" if row["train_months"] >= MIN_TRAIN_MONTHS and row["test_months"] > 0 else "skip"
    fold_plan_rows.append(row)
    if row["status"] == "run":
        valid_fold_specs.append(spec)
    del train_df, test_df
    gc.collect()

fold_plan_df = pd.DataFrame(fold_plan_rows)
fold_plan_df.to_csv(OUT_DIR / "v87_fold_plan.csv", index=False)
display_df(fold_plan_df, 20)
if len(valid_fold_specs) == 0:
    raise ValueError("no valid fold")

## 4. 单模型月度评估与成对稳定性

In [ ]:
def model_params(feature_fraction, seed):
    params = dict(BASE_PARAMS_FF10)
    params["feature_fraction"] = float(feature_fraction)
    params["seed"] = int(seed)
    params["feature_fraction_seed"] = int(seed)
    params["bagging_seed"] = int(seed)
    params["data_random_seed"] = int(seed)
    params["num_threads"] = int(NUM_THREADS)
    return params


def evaluate_predictions(test_meta, predictions, fold_id, seed, feature_fraction):
    panel = test_meta.copy()
    panel["score"] = np.asarray(predictions, dtype=float)
    rows = []
    selected = {}
    groups = panel.groupby(DATE_COL)
    for rebalance_date, month_df in progress_iter(groups, total=panel[DATE_COL].nunique(), desc="monthly metrics %s s%s %s" % (fold_id, seed, fraction_tag(feature_fraction)), leave=False):
        month_df = month_df.replace([np.inf, -np.inf], np.nan).dropna(subset=[TARGET_COL, "score"]).copy()
        n = int(len(month_df))
        if n < 3:
            continue
        true_n = _bi.min(TRUE_TOP_N, n)
        true_index = set(month_df.sort_values(TARGET_COL, ascending=False).head(true_n).index)
        sorted_score = month_df.sort_values("score", ascending=False)
        target_index = board_capped_indices(sorted_score, PRED_TOP_K)
        target_df = month_df.loc[target_index]
        target_set = set(target_df[STOCK_COL].tolist())
        selected[pd.Timestamp(rebalance_date)] = target_set
        hits = int(len(true_index.intersection(set(target_index))))
        binary_true = month_df.index.to_series().isin(true_index).astype(int).values
        top8_alpha = float(target_df[TARGET_COL].mean()) if len(target_df) else np.nan
        universe_alpha = float(month_df[TARGET_COL].mean())
        top_decile_n = _bi.max(1, int(round(n * 0.10)))
        top_decile_alpha = float(sorted_score.head(top_decile_n)[TARGET_COL].mean())
        rows.append({
            "fold_id": fold_id,
            "seed": int(seed),
            "feature_fraction": float(feature_fraction),
            "fraction_tag": fraction_tag(feature_fraction),
            "rebalance_date": pd.Timestamp(rebalance_date),
            "n": n,
            "rank_ic": safe_rank_ic(month_df[TARGET_COL], month_df["score"]),
            "auc_true_top20": binary_rank_auc(binary_true, month_df["score"].values),
            "precision_at8_true20": hits / float(len(target_index)) if len(target_index) else np.nan,
            "recall_at8_true20": hits / float(true_n) if true_n else np.nan,
            "ndcg_at8_true20": binary_ndcg_at_k(binary_true, month_df["score"].values, PRED_TOP_K),
            "top8_alpha": top8_alpha,
            "top8_edge": top8_alpha - universe_alpha if not pd.isnull(top8_alpha) else np.nan,
            "top_decile_edge": top_decile_alpha - universe_alpha,
        })
    return rows, selected


def prediction_stability_rows(test_meta, pred_a, pred_b, selected_a, selected_b, fold_id, seed, candidate_tag):
    panel = test_meta[[DATE_COL, STOCK_COL]].copy()
    panel["score_ff100"] = np.asarray(pred_a, dtype=float)
    panel["score_candidate"] = np.asarray(pred_b, dtype=float)
    rows = []
    groups = panel.groupby(DATE_COL)
    for rebalance_date, month_df in progress_iter(groups, total=panel[DATE_COL].nunique(), desc="prediction stability %s s%s" % (fold_id, seed), leave=False):
        date_value = pd.Timestamp(rebalance_date)
        set_a = selected_a.get(date_value, set())
        set_b = selected_b.get(date_value, set())
        union = set_a.union(set_b)
        rows.append({
            "fold_id": fold_id,
            "seed": int(seed),
            "rebalance_date": date_value,
            "candidate_tag": candidate_tag,
            "score_rank_corr": safe_rank_ic(month_df["score_ff100"], month_df["score_candidate"]),
            "top8_overlap_count": int(len(set_a.intersection(set_b))),
            "top8_overlap_ratio": len(set_a.intersection(set_b)) / float(PRED_TOP_K),
            "top8_jaccard": len(set_a.intersection(set_b)) / float(len(union)) if len(union) else np.nan,
        })
    return rows

## 5. 运行 5 folds × 3 seeds × 4 fractions

In [ ]:
monthly_rows = []
stability_rows = []
importance_rows = []
model_meta_rows = []

for spec in progress_iter(valid_fold_specs, total=len(valid_fold_specs), desc="walk-forward folds"):
    fold_id = spec["fold_id"]
    train_df, test_df = make_fold_frames(df_all, spec)
    feature_cols, removed_cols = select_features_train_only(train_df)
    fill_values = train_df[feature_cols].median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(fill_values).fillna(0).astype(np.float32)
    y_train = train_df[TARGET_COL].astype(float)
    X_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(fill_values).fillna(0).astype(np.float32)
    test_meta = test_df[[DATE_COL, STOCK_COL, TARGET_COL]].copy().reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    dtrain = lgb.Dataset(X_train, label=y_train, feature_name=list(feature_cols), free_raw_data=False)
    if hasattr(dtrain, "construct"):
        dtrain.construct()

    for seed in progress_iter(SEEDS, total=len(SEEDS), desc="seeds %s" % fold_id, leave=False):
        predictions = {}
        selected_by_fraction = {}
        for ff in progress_iter(FEATURE_FRACTIONS, total=len(FEATURE_FRACTIONS), desc="feature fractions %s s%s" % (fold_id, seed), leave=False):
            params = model_params(ff, seed)
            model = lgb.train(params, dtrain, num_boost_round=int(FIXED_ITER))
            pred = np.asarray(model.predict(X_test[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
            metric_one, selected_one = evaluate_predictions(test_meta, pred, fold_id, seed, ff)
            monthly_rows.extend(metric_one)
            predictions[fraction_tag(ff)] = pred
            selected_by_fraction[fraction_tag(ff)] = selected_one

            gain = np.asarray(model.feature_importance(importance_type="gain"), dtype=float)
            gain_total = float(gain.sum())
            for i, feature in progress_iter(enumerate(feature_cols), total=len(feature_cols), desc="importance %s s%s %s" % (fold_id, seed, fraction_tag(ff)), leave=False):
                importance_rows.append({
                    "fold_id": fold_id,
                    "seed": int(seed),
                    "feature_fraction": float(ff),
                    "fraction_tag": fraction_tag(ff),
                    "feature": feature,
                    "importance_gain": float(gain[i]),
                    "importance_gain_pct": float(gain[i] / gain_total) if gain_total > 0 else np.nan,
                })
            model_meta_rows.append({
                "fold_id": fold_id,
                "seed": int(seed),
                "feature_fraction": float(ff),
                "fraction_tag": fraction_tag(ff),
                "cutoff": spec["cutoff"],
                "test_start": spec["test_start"],
                "test_end": spec["test_end"],
                "train_rows": int(len(train_df)),
                "train_months": int(train_df[DATE_COL].nunique()),
                "test_rows": int(len(test_df)),
                "test_months": int(test_df[DATE_COL].nunique()),
                "feature_count": int(len(feature_cols)),
                "removed_feature_count": int(len(removed_cols)),
                "removed_features": ",".join(removed_cols),
            })
            del model
            gc.collect()

        if "ff100" in predictions:
            candidate_tags = [fraction_tag(x) for x in FEATURE_FRACTIONS if fraction_tag(x) != "ff100"]
            for candidate_tag in progress_iter(candidate_tags, total=len(candidate_tags), desc="fraction stability %s s%s" % (fold_id, seed), leave=False):
                stability_rows.extend(prediction_stability_rows(
                    test_meta,
                    predictions["ff100"], predictions[candidate_tag],
                    selected_by_fraction["ff100"], selected_by_fraction[candidate_tag],
                    fold_id, seed, candidate_tag,
                ))
        del predictions, selected_by_fraction
        gc.collect()

    del dtrain, X_train, X_test, y_train, test_meta, train_df, test_df
    gc.collect()

monthly_metrics_df = pd.DataFrame(monthly_rows)
prediction_stability_df = pd.DataFrame(stability_rows)
feature_importance_df = pd.DataFrame(importance_rows)
model_meta_df = pd.DataFrame(model_meta_rows)

monthly_metrics_df.to_csv(OUT_DIR / "v87_monthly_metrics.csv", index=False)
prediction_stability_df.to_csv(OUT_DIR / "v87_prediction_stability_monthly.csv", index=False)
feature_importance_df.to_csv(OUT_DIR / "v87_feature_importance.csv", index=False)
model_meta_df.to_csv(OUT_DIR / "v87_model_meta.csv", index=False)
print("monthly metrics:", monthly_metrics_df.shape)
display_df(model_meta_df, 20)

## 6. 成对比较、种子/年度稳定性与预注册判定

In [ ]:
METRIC_COLS = [
    "rank_ic", "auc_true_top20", "precision_at8_true20", "recall_at8_true20",
    "ndcg_at8_true20", "top8_alpha", "top8_edge", "top_decile_edge",
]


def summarize_metrics(df, group_cols):
    rows = []
    grouped = list(df.groupby(group_cols))
    for keys, part in progress_iter(grouped, total=len(grouped), desc="summarize metrics"):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict((group_cols[i], keys[i]) for i in range(len(group_cols)))
        row["months"] = int(len(part))
        for metric in METRIC_COLS:
            values = pd.to_numeric(part[metric], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            row[metric + "_mean"] = float(values.mean()) if len(values) else np.nan
            row[metric + "_median"] = float(values.median()) if len(values) else np.nan
            row[metric + "_positive_rate"] = float((values > 0).mean()) if len(values) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


fraction_summary_df = summarize_metrics(monthly_metrics_df, ["fraction_tag"])
seed_summary_df = summarize_metrics(monthly_metrics_df, ["seed", "fraction_tag"])
fold_summary_df = summarize_metrics(monthly_metrics_df, ["fold_id", "fraction_tag"])
yearly_summary_df = summarize_metrics(monthly_metrics_df, ["fold_id", "seed", "fraction_tag"])

key_cols = ["fold_id", "seed", "rebalance_date"]


def build_pairwise_against_ff100(monthly_df, candidate_tag):
    base = monthly_df[monthly_df["fraction_tag"] == "ff100"][key_cols + METRIC_COLS].copy()
    candidate = monthly_df[monthly_df["fraction_tag"] == candidate_tag][key_cols + METRIC_COLS].copy()
    base = base.rename(columns=dict((c, c + "_ff100") for c in METRIC_COLS))
    candidate = candidate.rename(columns=dict((c, c + "_candidate") for c in METRIC_COLS))
    paired = base.merge(candidate, on=key_cols, how="inner")
    paired["candidate_tag"] = candidate_tag
    for metric in METRIC_COLS:
        paired[metric + "_delta"] = paired[metric + "_candidate"] - paired[metric + "_ff100"]
    return paired


candidate_tags = [tag for tag in FRACTION_TAGS if tag != "ff100"]
pairwise_parts = []
for candidate_tag in progress_iter(candidate_tags, total=len(candidate_tags), desc="pair fractions vs ff100"):
    pairwise_parts.append(build_pairwise_against_ff100(monthly_metrics_df, candidate_tag))
all_pairwise_monthly_df = pd.concat(pairwise_parts, ignore_index=True)
pairwise_monthly_df = all_pairwise_monthly_df[all_pairwise_monthly_df["candidate_tag"] == "ff080"].copy()


def summarize_pairwise(df, group_cols):
    rows = []
    grouped = list(df.groupby(group_cols)) if len(group_cols) else [((), df)]
    for keys, part in progress_iter(grouped, total=len(grouped), desc="summarize paired deltas"):
        if len(group_cols) == 0:
            keys = ()
        elif not isinstance(keys, tuple):
            keys = (keys,)
        row = dict((group_cols[i], keys[i]) for i in range(len(group_cols)))
        row["paired_months"] = int(len(part))
        for metric in METRIC_COLS:
            values = pd.to_numeric(part[metric + "_delta"], errors="coerce").dropna()
            row[metric + "_delta_mean"] = float(values.mean()) if len(values) else np.nan
            row[metric + "_delta_median"] = float(values.median()) if len(values) else np.nan
            row[metric + "_delta_positive_rate"] = float((values > 0).mean()) if len(values) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


pairwise_seed_df = summarize_pairwise(pairwise_monthly_df, ["seed"])
pairwise_fold_df = summarize_pairwise(pairwise_monthly_df, ["fold_id"])
pairwise_fold_seed_df = summarize_pairwise(pairwise_monthly_df, ["fold_id", "seed"])
pairwise_overall_df = summarize_pairwise(pairwise_monthly_df, [])
all_pairwise_overall_df = summarize_pairwise(all_pairwise_monthly_df, ["candidate_tag"])
all_pairwise_seed_df = summarize_pairwise(all_pairwise_monthly_df, ["candidate_tag", "seed"])
all_pairwise_fold_df = summarize_pairwise(all_pairwise_monthly_df, ["candidate_tag", "fold_id"])

overall = pairwise_overall_df.iloc[0]
seed_wins = int((pairwise_seed_df["rank_ic_delta_mean"] > 0).sum())
fold_wins = int((pairwise_fold_df["rank_ic_delta_mean"] > 0).sum())
worst_seed_delta = float(pairwise_seed_df["rank_ic_delta_mean"].min())
rank_pass = bool(overall["rank_ic_delta_mean"] > 0)
edge_pass = bool(overall["top8_edge_delta_mean"] >= 0)
seed_pass = bool(seed_wins >= _bi.min(MIN_SEED_WINS, len(pairwise_seed_df)))
fold_pass = bool(fold_wins >= _bi.min(MIN_FOLD_WINS, len(pairwise_fold_df)))
worst_seed_pass = bool(worst_seed_delta >= MAX_WORST_SEED_RANK_IC_DELTA)
adopt = bool(rank_pass and edge_pass and seed_pass and fold_pass and worst_seed_pass)

decision_df = pd.DataFrame([{
    "decision": "adopt_feature_fraction_0_8" if adopt else "keep_1_0_or_collect_more_evidence",
    "rank_ic_delta_mean": overall["rank_ic_delta_mean"],
    "top8_edge_delta_mean": overall["top8_edge_delta_mean"],
    "precision_at8_delta_mean": overall["precision_at8_true20_delta_mean"],
    "ndcg_at8_delta_mean": overall["ndcg_at8_true20_delta_mean"],
    "seed_wins": seed_wins,
    "seed_total": int(len(pairwise_seed_df)),
    "fold_wins": fold_wins,
    "fold_total": int(len(pairwise_fold_df)),
    "worst_seed_rank_ic_delta": worst_seed_delta,
    "rank_pass": rank_pass,
    "top8_edge_pass": edge_pass,
    "seed_pass": seed_pass,
    "fold_pass": fold_pass,
    "worst_seed_pass": worst_seed_pass,
    "mean_score_rank_corr": safe_mean(prediction_stability_df[prediction_stability_df["candidate_tag"] == "ff080"]["score_rank_corr"]),
    "mean_top8_overlap_ratio": safe_mean(prediction_stability_df[prediction_stability_df["candidate_tag"] == "ff080"]["top8_overlap_ratio"]),
}])

fraction_decision_rows = []
for candidate_tag in progress_iter(candidate_tags, total=len(candidate_tags), desc="build fraction decision table"):
    candidate_overall = all_pairwise_overall_df[all_pairwise_overall_df["candidate_tag"] == candidate_tag].iloc[0]
    candidate_seed = all_pairwise_seed_df[all_pairwise_seed_df["candidate_tag"] == candidate_tag]
    candidate_fold = all_pairwise_fold_df[all_pairwise_fold_df["candidate_tag"] == candidate_tag]
    candidate_stability = prediction_stability_df[prediction_stability_df["candidate_tag"] == candidate_tag]
    candidate_seed_wins = int((candidate_seed["rank_ic_delta_mean"] > 0).sum())
    candidate_fold_wins = int((candidate_fold["rank_ic_delta_mean"] > 0).sum())
    candidate_worst_seed = float(candidate_seed["rank_ic_delta_mean"].min())
    fraction_decision_rows.append({
        "candidate_tag": candidate_tag,
        "baseline_tag": "ff100",
        "rank_ic_delta_mean": candidate_overall["rank_ic_delta_mean"],
        "top8_edge_delta_mean": candidate_overall["top8_edge_delta_mean"],
        "precision_at8_delta_mean": candidate_overall["precision_at8_true20_delta_mean"],
        "ndcg_at8_delta_mean": candidate_overall["ndcg_at8_true20_delta_mean"],
        "seed_wins": candidate_seed_wins,
        "seed_total": int(len(candidate_seed)),
        "fold_wins": candidate_fold_wins,
        "fold_total": int(len(candidate_fold)),
        "worst_seed_rank_ic_delta": candidate_worst_seed,
        "mean_score_rank_corr": safe_mean(candidate_stability["score_rank_corr"]),
        "mean_top8_overlap_ratio": safe_mean(candidate_stability["top8_overlap_ratio"]),
        "passes_same_guardrails": bool(
            candidate_overall["rank_ic_delta_mean"] > 0
            and candidate_overall["top8_edge_delta_mean"] >= 0
            and candidate_seed_wins >= _bi.min(MIN_SEED_WINS, len(candidate_seed))
            and candidate_fold_wins >= _bi.min(MIN_FOLD_WINS, len(candidate_fold))
            and candidate_worst_seed >= MAX_WORST_SEED_RANK_IC_DELTA
        ),
        "is_pre_registered_primary": bool(candidate_tag == "ff080"),
    })
fraction_decision_df = pd.DataFrame(fraction_decision_rows)

outputs = {
    "v87_fraction_summary.csv": fraction_summary_df,
    "v87_seed_summary.csv": seed_summary_df,
    "v87_fold_summary.csv": fold_summary_df,
    "v87_yearly_seed_summary.csv": yearly_summary_df,
    "v87_pairwise_monthly.csv": pairwise_monthly_df,
    "v87_pairwise_summary_by_seed.csv": pairwise_seed_df,
    "v87_pairwise_summary_by_fold.csv": pairwise_fold_df,
    "v87_pairwise_summary_by_fold_seed.csv": pairwise_fold_seed_df,
    "v87_pairwise_summary_overall.csv": pairwise_overall_df,
    "v87_pre_registered_decision.csv": decision_df,
    "v87_all_fraction_pairwise_monthly.csv": all_pairwise_monthly_df,
    "v87_all_fraction_summary_overall.csv": all_pairwise_overall_df,
    "v87_all_fraction_summary_by_seed.csv": all_pairwise_seed_df,
    "v87_all_fraction_summary_by_fold.csv": all_pairwise_fold_df,
    "v87_fraction_decision_table.csv": fraction_decision_df,
}
for filename, frame in progress_iter(list(outputs.items()), total=len(outputs), desc="save summary tables"):
    frame.to_csv(OUT_DIR / filename, index=False)

print("pre-registered decision")
display_df(decision_df, 10)
display_df(fraction_decision_df, 10)
display_df(pairwise_seed_df, 10)
display_df(pairwise_fold_df, 10)

## 7. 可视化

In [ ]:
def save_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(str(path), dpi=140, bbox_inches="tight")
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)
    print("saved figure:", path)


# 1. Fold-level OOS comparison.
fold_plot = fold_summary_df.copy()
folds = [x["fold_id"] for x in valid_fold_specs]
positions = np.arange(len(folds))
plot_tags = FRACTION_TAGS
plot_colors = ["#59636d", "#3f75a2", "#2d8f78", "#d4543c"]
width = 0.18
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
for i, tag in progress_iter(enumerate(plot_tags), total=len(plot_tags), desc="plot fold metrics", leave=False):
    part = fold_plot[fold_plot["fraction_tag"] == tag].set_index("fold_id").reindex(folds)
    offset = (i - (len(plot_tags) - 1) / 2.0) * width
    axes[0].bar(positions + offset, part["rank_ic_mean"].values, width=width, label=tag, color=plot_colors[i])
    axes[1].bar(positions + offset, part["top8_edge_mean"].values, width=width, label=tag, color=plot_colors[i])
axes[0].axhline(0, color="#333333", linewidth=0.8)
axes[1].axhline(0, color="#333333", linewidth=0.8)
axes[0].set_title("OOS RankIC by fold")
axes[1].set_title("OOS Top8 board-cap edge by fold")
axes[1].set_xticks(positions)
axes[1].set_xticklabels(folds)
axes[0].legend(loc="best")
axes[1].legend(loc="best")
save_figure(fig, "v87_fold_oos_comparison.png")


# 2. Overall paired deltas for every candidate against ff100.
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
delta_plot = all_pairwise_overall_df.set_index("candidate_tag").reindex(candidate_tags)
axes[0].bar(np.arange(len(candidate_tags)), delta_plot["rank_ic_delta_mean"].values, color=plot_colors[1:])
axes[0].axhline(0, color="#333333", linewidth=0.8)
axes[0].set_xticks(np.arange(len(candidate_tags)))
axes[0].set_xticklabels(candidate_tags)
axes[0].set_title("Mean RankIC delta vs ff100")
axes[1].bar(np.arange(len(candidate_tags)), delta_plot["top8_edge_delta_mean"].values, color=plot_colors[1:])
axes[1].axhline(0, color="#333333", linewidth=0.8)
axes[1].set_xticks(np.arange(len(candidate_tags)))
axes[1].set_xticklabels(candidate_tags)
axes[1].set_title("Mean Top8 edge delta vs ff100")
save_figure(fig, "v87_paired_rankic_deltas.png")


# 3. Prediction and holding stability.
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
stability_by_candidate = []
for candidate_tag, part in progress_iter(list(prediction_stability_df.groupby("candidate_tag")), total=prediction_stability_df["candidate_tag"].nunique(), desc="plot prediction stability", leave=False):
    stability_by_candidate.append({
        "candidate_tag": candidate_tag,
        "score_rank_corr": safe_mean(part["score_rank_corr"]),
        "top8_overlap_ratio": safe_mean(part["top8_overlap_ratio"]),
    })
stability_plot = pd.DataFrame(stability_by_candidate).set_index("candidate_tag").reindex(candidate_tags)
axes[0].bar(np.arange(len(candidate_tags)), stability_plot["score_rank_corr"].values, color=plot_colors[1:])
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Score rank correlation vs ff100")
axes[1].bar(np.arange(len(candidate_tags)), stability_plot["top8_overlap_ratio"].values, color=plot_colors[1:])
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Top8 holding overlap vs ff100")
for ax in axes:
    ax.set_xticks(np.arange(len(candidate_tags)))
    ax.set_xticklabels(candidate_tags)
save_figure(fig, "v87_prediction_and_holding_stability.png")


# 4. Aggregate feature importance and concentration.
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
flat_axes = axes.ravel()
for i, tag in progress_iter(enumerate(plot_tags), total=len(plot_tags), desc="plot importance", leave=False):
    part = feature_importance_df[feature_importance_df["fraction_tag"] == tag]
    gain = part.groupby("feature")["importance_gain_pct"].mean().sort_values(ascending=False).head(15).sort_values()
    flat_axes[i].barh(np.arange(len(gain)), gain.values, color=plot_colors[i])
    flat_axes[i].set_yticks(np.arange(len(gain)))
    flat_axes[i].set_yticklabels(gain.index, fontsize=9)
    flat_axes[i].set_title("%s mean gain share" % tag)
    flat_axes[i].set_xlabel("gain share")
save_figure(fig, "v87_feature_importance_comparison.png")

## 8. 输出说明

In [ ]:
feature_manifest_df = pd.DataFrame({"feature": FULL_V46_COLS})
feature_manifest_df.to_csv(OUT_DIR / "v87_feature_manifest.csv", index=False)

readme_lines = [
    "V87 V46 feature_fraction robustness experiment",
    "Run timestamp: %s" % RUN_TIMESTAMP,
    "Data: %s" % DATA_PATH,
    "Only feature_fraction changes: 1.0, 0.9, 0.8, 0.7.",
    "All folds are label_end_safe and use exact V46 base parameters with fixed120.",
    "Primary result: v87_pre_registered_decision.csv",
    "All-fraction comparison: v87_fraction_decision_table.csv",
    "Pairwise evidence: v87_pairwise_summary_by_seed.csv and v87_pairwise_summary_by_fold.csv",
    "This is a fast walk-forward ranking experiment, not an execution backtest.",
]
with open(OUT_DIR / "v87_README.txt", "w") as f:
    f.write("\n".join(readme_lines))

print("saved outputs:")
for path in _bi.sorted(OUT_DIR.glob("*.csv")):
    print("-", path)
print("figures:")
for path in _bi.sorted(FIGURE_DIR.glob("*.png")):
    print("-", path)
print("primary result:")
display_df(decision_df, 10)
display_df(fraction_decision_df, 10)